# Pay SR3 when the data says the Fed is turning hawkish. Does it work?

JWS Macro #8 (23-Aug-2026) reads a chart and says:

> *"we see a tidy 5 week lead of data surprise vs. Fedspeak"*

and the trade that follows is the simplest one in macro: **if the economic
data tells you what the committee is about to sound like, pay the front end
when the data runs hot and receive it when the data softens.**

Three studies on `main` have already taken the measuring end of this apart.
The lead is **+11 to +13 weeks on 2023-2026, not five**; that window is a
twenty-year maximum; twenty-one years of a second judge model leaves **+2
weeks at r 0.12**; and asking whether the *gap* between the two sides trades
gave 2048 dead cells. None of them asked the plainest question of all, which
is the one here: **not the gap, and not the lead — the direction of the data
itself, traded outright in SR3.**

## What was measured -- the numbers up front

*(Every figure here is restated by a cell below, and the last cell of the
notebook prints all of them together in exactly the form the prose quotes them.
If the cells and this summary ever disagree, the cells are what ran.
`_audit_fed_expected_sentiment_numbers.py` enforces that as part of the build.)*

**The answer is no, in all three samples, and the searched maximum is below
the null MEDIAN in every one of them.**

1. **The SR3 grid is dead, and the searched maximum is below the null MEDIAN.**
   **672** cells / **1344** trials over **433** weeks of SR3. Best cell
   `chg/L11/thr0.0/h8/pack1/follow` at a weekly Sharpe of **0.1090**, against a
   rotation null whose median is **0.1155** and whose 95th percentile is
   **0.1500**: the observation sits at the **33.5%** percentile of its own null
   and **p = 0.6657** on an exhaustive **355**-rotation test with a floor of
   **0.0028**. Deflated Sharpe **0.0080** against an SR0 of **0.1957**. A
   Romano-Wolf stepdown over the whole family rejects **0** of **669** scoreable
   cells.
2. **Twenty-one years makes it worse, not better -- and it flips sign.** The 2y
   SOFR OIS over **1122** weeks from 2005. The pre-registered cell earns
   **-1.4686bp** per trade over **280** trades, **-411.19bp** in total, t
   **-1.0540**; the always-on weekly book loses **-47.77bp**. The grid's best is
   **0.0561** against a null median of **0.0610**, **p = 0.6861** on **1044**
   exhaustive rotations, DSR **0.2707**. The same cell measured on the 2018+
   SR3-era subsample is *positive*; extending the sample flips it. That is what
   nothing looks like. **This is also the clean re-run of the "does it reach the
   price" sections that `fed_sentiment_lead` and `fedlock_sentiment_lead` both
   computed on the poisoned `RATES.OIS.USD_SOFR.PAR.2Y` tag and recorded as "not
   re-run"** -- the answer does not change, and now it rests on a series that is
   a rate.
3. **The window where the lead is strongest is the deadest of the three.**
   On the **121** weeks where the point-in-time Fed sentiment index exists --
   the window in which two prior studies measured an +11 to +13 week lead with
   a correlation above 0.7 -- the always-on weekly book earns **+0.75bp** in
   total. The grid's
   best is **0.2038** against a null median of **0.2069**, **p = 0.5682** on
   **43** rotations, DSR **0.1490**.
4. **The roll is the trap, and it is worth more than the signal.** Measured on
   **432** Fridays of which **33** contain a contract change: differencing a
   fixed-rank price column -- the obvious construction -- gives a signed mean
   weekly move of **+4.56bp** on roll weeks against **-0.81bp** otherwise, and
   an absolute median of **16.5bp** against **5.0bp**. The change in the
   contract actually HELD over those same weeks is **-2.03bp**. The gap is
   **+6.59bp** per roll week and **+217.5bp** in total, pure fabrication --
   larger than the whole always-on book's **+97.25bp**. Off the roll weeks the
   two constructions agree to **0.0bp**, which is the known answer that
   certifies the measurement. Nothing in this study differences two contracts:
   the worst disagreement between a booked mark and the settle panel, anywhere,
   is **0.0**.
5. **The alignment is arithmetic, not a swept parameter.** If the data leads
   Fedspeak by L weeks, a position held h weeks is forecast by
   `C_{t-(L-h)} - C_{t-L}` -- two lags, both pinned by (L, h). Writing the
   signal as `zc.shift(k)` and sweeping k is the alignment for MEASURING a
   lead, not for trading it, and it makes the signal gratuitously stale. The
   grid's only signal axis is L, taking four values somebody has defended.
6. **The vintage exposure is bounded, and bounding it kills the result rather
   than confirming it.** The Citi surprise snapshot is a single vintage with no
   publication axis -- the largest un-gated look-ahead in the stack. Delaying
   the composite one extra week moves the 21-year cell from **-411.19bp** to
   **+198.44bp** -- a 610bp swing that changes its sign -- the SR3 cell from
   **+152.50bp** to **+60.50bp**, and the 121-week cell from **+47.50bp** to
   **+196.50bp**. A result that quadruples, quarters, or changes sign when its
   own input is delayed by one week was never a result.
7. **The pre-registered cell, written down before the grid ran.** `chg`,
   lead 0, four weeks, always-on, third deferred SR3, follow: **108** trades at
   **+1.4120bp** each, **+152.50bp** in total, hit **50.9%**, t **0.5229**,
   shared sign-flip **p = 0.6059**, and its 108 trades live in **51** episodes.
   The literal always-on weekly reading of the question -- receive when the data
   softens, pay when it firms, re-decided every Friday -- earns **+147.00bp**
   gross over **432** weeks, pays **49.75bp** of turnover for a net
   **+97.25bp**, at an annualised Sharpe of **0.1169** and a maximum drawdown of
   **-258.5bp**. Eight years, one third of a basis point a week, and a drawdown
   two and a half times the total.
8. **The direction the search picks is at least the right one, which is not
   the same as being a signal.** Across the SR3 grid **48.8%** of cells prefer
   `follow` -- pay when the data runs hot -- and among the 50 best cells that
   rises to **60.0%**. So what edge there is leans the way the claim says it
   should. It is still smaller than the null's median.

In [1]:
from __future__ import annotations

import dataclasses
import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype+notebook_connected"
T0 = time.time()

HERE = Path.cwd()
REPO = HERE if (HERE / "MDP").exists() else HERE.parents[1]
for _p in (str(REPO), str(REPO / "notebooks" / "rv")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import fed_detachment_prices as PX  # noqa: E402
import fed_expected_sentiment as E  # noqa: E402
import fed_expected_sentiment_run as R  # noqa: E402

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

BG, GRID_C = "#11151c", "#2a3340"
GREY, RED, BLUE, AMBER, GREEN = "#8b97a8", "#e05a52", "#5aa9e0", "#e0a83a", "#4fbf7f"


def style(fig, height=520, title=None):
    fig.update_layout(template="plotly_dark", height=height, title=title,
                      paper_bgcolor=BG, plot_bgcolor=BG,
                      margin=dict(l=60, r=30, t=60 if title else 24, b=48),
                      legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom"))
    fig.update_xaxes(gridcolor=GRID_C, zerolinecolor=GRID_C)
    fig.update_yaxes(gridcolor=GRID_C, zerolinecolor=GRID_C)
    return fig


RES = pickle.load(open(REPO / "notebooks" / "rv" /
                       "fed_expected_sentiment_results.pkl", "rb"))
print("samples:", list(RES))

samples: ['SR3', 'OIS21', 'JPM']


## 1. The alignment, which is the one piece of arithmetic that decides the study

It is tempting to write the signal as `zc.shift(k)` and sweep `k`. That is the
alignment for *measuring* a lead, not for *trading* it, and it makes the signal
gratuitously stale.

Do the arithmetic. If the composite `C` leads Fedspeak `S` by `L` weeks so that
`S_t` tracks `C_{t-L}`, then a position opened at `t` and held `h` weeks is
exposed to `S_{t+h}`, which tracks `C_{t+h-L} = C_{t-(L-h)}`. So the forecast of
the change in the Fed's tone over the holding period is

> `s_t = C_{t-(L-h)} - C_{t-L}`

Both ends are lags and both are pinned by `(L, h)`. There is no free parameter,
and the grid's only signal axis is `L` — four values each of which somebody has
defended: **0** (no lead: trade the direction the data has already moved),
**2** (the 21-year survivor), **5** (JWS's claim), **11** (what 2023-2026
measures).

In [2]:
rows = []
for lw in E.LEADS_W:
    for h in E.HORIZONS_W:
        c = dataclasses.replace(E.PRIMARY, lead_w=lw, horizon_w=h)
        near, far = c.lags()
        rows.append({"lead_L": lw, "horizon_h": h, "lag_near": near,
                     "lag_far": far, "window_w": far - near})
LAGS = pd.DataFrame(rows)
print("the (L, h) -> (near, far) map, weeks:\n")
print(LAGS.pivot(index="lead_L", columns="horizon_h",
                 values=["lag_near", "lag_far"]).to_string())
print(f"\nlag_near is never negative: {bool((LAGS.lag_near >= 0).all())}")
print(f"lag_far always exceeds lag_near: {bool((LAGS.lag_far > LAGS.lag_near).all())}")
print(f"\nPRE-REGISTERED PRIMARY: {E.PRIMARY.label()}")
print(E.PRIMARY.describe().to_string(index=False))

the (L, h) -> (near, far) map, weeks:

          lag_near          lag_far            
horizon_h        1  2  4  8       1   2   4   8
lead_L                                         
0                0  0  0  0       1   2   4   8
2                1  0  0  0       2   2   2   2
5                4  3  1  0       5   5   5   5
11              10  9  7  3      11  11  11  11

lag_near is never negative: True
lag_far always exceeds lag_near: True

PRE-REGISTERED PRIMARY: chg/L0/h4/thr0/out3/follow [lags 0,4]
                        knob    value
                     reading      chg
                      lead_w        0
                reg_window_w      104
                   horizon_w        4
                   threshold      0.0
                   structure     out3
                   direction        1
          entry_lag_sessions        1
             sent_z_window_w       52
                sent_z_min_w       26
             cost_bp_one_way     0.25
              rotation_draws     5

## 2. The roll, measured on a construction whose answer is known

A weekly directional futures book is exactly the thing a naive roll destroys.
The obvious construction — build a "rank 3 price" column and difference it —
differences **two different contracts** on every roll week.

This is run FIRST, before any book is scored, because a checking tool that is
itself wrong reports success. The known answer is the second half: on a week
where the rank did *not* change contract, the naive and the correct
construction are the same arithmetic and must agree to **0.0bp**. They do. On
the roll weeks they do not, and the gap is the fabrication.

In [3]:
sr3 = RES["SR3"]
rp = sr3["roll_placebo"]
ROLL = pd.Series({k: v for k, v in rp.items() if k != "series"})
print("SR3 rank 3, weekly, 2018-05-11..2026-08-21\n")
for k, v in ROLL.items():
    print(f"  {k:32s} {v if not isinstance(v, float) else round(v, 4)}")
print(f"\n  the naive construction fabricates "
      f"{rp['fabricated_mean_bp']:+.2f}bp per roll week and "
      f"{rp['fabricated_total_bp']:+.1f}bp in total,")
print(f"  and exactly {rp['fabricated_on_flat_weeks_bp']:.1f}bp on the "
      f"{rp['weeks'] - rp['roll_weeks']} weeks with no contract change.")

ser = rp["series"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=ser.index[~ser["rolled"]],
                         y=ser.loc[~ser["rolled"], "d_naive_bp"], mode="markers",
                         name="no roll", marker=dict(color=GREY, size=4)))
fig.add_trace(go.Scatter(x=ser.index[ser["rolled"]],
                         y=ser.loc[ser["rolled"], "d_naive_bp"], mode="markers",
                         name="roll week (naive: two contracts)",
                         marker=dict(color=RED, size=9, symbol="x")))
fig.add_trace(go.Scatter(x=ser.index[ser["rolled"]],
                         y=ser.loc[ser["rolled"], "d_true_bp"], mode="markers",
                         name="roll week (correct: one contract)",
                         marker=dict(color=GREEN, size=9)))
style(fig, 460, "Weekly change in a rank-3 SR3 mark: what the roll fabricates")
fig.update_yaxes(title="bp")
fig.show()

SR3 rank 3, weekly, 2018-05-11..2026-08-21

  rank                             3.0
  weeks                            432.0
  roll_weeks                       33.0
  naive_roll_mean_bp               4.5606
  naive_flat_mean_bp               -0.8083
  naive_roll_abs_median_bp         16.5
  naive_flat_abs_median_bp         5.0
  true_roll_mean_bp                -2.0303
  true_roll_abs_median_bp          5.0
  fabricated_mean_bp               6.5909
  fabricated_total_bp              217.5
  fabricated_on_flat_weeks_bp      0.0

  the naive construction fabricates +6.59bp per roll week and +217.5bp in total,
  and exactly 0.0bp on the 399 weeks with no contract change.


**Nothing in this study ever differences two contracts.** The discrete book
fixes the symbol at the filled entry and re-reads that same symbol at exit; the
always-on book prices each week on the contract held that week and pays a full
round trip when the contract changes. `gate_no_roll_jump` checks every priced
row against the settle panel: each row's two marks must both be the contract it
names.

In [4]:
GATES = []
for name, res in RES.items():
    for which in ("discrete", "weekly"):
        g = res["primary"][f"roll_gate_{which}"]
        GATES.append({"sample": name, "book": which, "rows": g.get("rows"),
                      "marks_checked": g.get("marks_checked"),
                      "worst_mark_diff": g.get("worst_mark_diff"),
                      "roll_weeks": g.get("roll_weeks")})
GATES = pd.DataFrame(GATES)
print(GATES.to_string(index=False))
print(f"\nG-X3: worst disagreement between a booked mark and the settle panel, "
      f"anywhere: {float(pd.to_numeric(GATES['worst_mark_diff'], errors='coerce').max()):.1f}")

sample     book  rows  marks_checked  worst_mark_diff  roll_weeks
   SR3 discrete   108            108              0.0         NaN
   SR3   weekly   432            432              0.0        33.0
 OIS21 discrete   280              0              0.0         NaN
 OIS21   weekly  1120              0              0.0         0.0
   JPM discrete    30             30              0.0         NaN
   JPM   weekly   120            120              0.0         9.0

G-X3: worst disagreement between a booked mark and the settle panel, anywhere: 0.0


## 3. The point-in-time gates

`G-X1` truncating the inputs after `t` must not move the signal at `t`.
`G-X2` the fill is the NEXT session after the Friday the signal is read from.
`G-X4` a rate is a rate — the 2y OIS leg is built from the CurveStore discount
factors, not from `RATES.OIS.USD_SOFR.PAR.2Y`, which in the shared tag cache is
47% swaption normal vol.

In [5]:
for name, res in RES.items():
    gt = res.get("gate_trailing")
    gf = res.get("gate_fill")
    print(f"{name}:")
    if gt is not None and len(gt):
        print(f"   G-X1  {len(gt)} probes, worst |diff| {float(gt['abs_diff'].max()):.1e}")
    if gf is not None and len(gf):
        print(f"   G-X2  {len(gf)} probes, every fill strictly after its signal "
              f"Friday: {bool(gf['strictly_after'].all())}")
    cg = res.get("coverage_gate")
    if cg is not None:
        print(f"   G-P3  every structure prices >= "
              f"{float(cg['share'].min()):.1%} of its weeks")
    if "rate_gate" in res:
        rg = res["rate_gate"]
        print(f"   G-X4  2y OIS: {rg['n']} days {rg['first'].date()}..{rg['last'].date()}, "
              f"range {rg['min']:.3f}%..{rg['max']:.3f}%, band {rg['band']}")

SR3:
   G-X1  24 probes, worst |diff| 0.0e+00
   G-X2  12 probes, every fill strictly after its signal Friday: True
   G-P3  every structure prices >= 99.8% of its weeks
OIS21:
   G-X1  24 probes, worst |diff| 0.0e+00
   G-X2  12 probes, every fill strictly after its signal Friday: True
   G-P3  every structure prices >= 99.8% of its weeks
   G-X4  2y OIS: 5516 days 2005-01-03..2026-08-21, range -0.017%..5.744%, band (-1.0, 15.0)
JPM:
   G-X1  36 probes, worst |diff| 0.0e+00
   G-X2  12 probes, every fill strictly after its signal Friday: True
   G-P3  every structure prices >= 99.2% of its weeks
   G-X4  2y OIS: 5516 days 2005-01-03..2026-08-21, range -0.017%..5.744%, band (-1.0, 15.0)


## 4. The three samples

They are never pooled. A Sharpe from one against a Sharpe from another compares
instruments **and** samples at once, and `project_global_cb_hawk_dove` measured
that the leg mix alone spanned a wider Sharpe range than the structure ranking
it was trying to read.

In [6]:
HEAD = pd.DataFrame([R.headline(r) for r in RES.values()])
print(HEAD.to_string(index=False))
print()
for k, v in RES.items():
    print(f"{k:6s}  {v['note']}")

sample  weeks      first       last  PRIMARY trades  PRIMARY avg_bp  PRIMARY sharpe_ann  PRIMARY t  PRIMARY sign_flip_p  WEEKLY total_bp  WEEKLY sharpe_ann  WEEKLY weeks_in_mkt  cells  trials                      best cell  best sharpe_wk  null median  null q95  p_rotation  p_floor      DSR  RW rejected
   SR3    433 2018-05-11 2026-08-21             108        1.412037            0.181434   0.522948             0.605870        97.250000           0.116904                  432    672    1344 chg/L11/thr0.0/h8/pack1/follow        0.108989     0.115490  0.149985    0.665730 0.002809 0.008047            0
 OIS21   1122 2005-02-25 2026-08-21             280       -1.468553           -0.227116  -1.054035             0.295885       -47.774336          -0.027051                 1120     96     192 chg/L11/thr0.0/h8/ois2y/follow        0.056144     0.061031  0.083563    0.686124 0.000957 0.270720            0
   JPM    121 2024-04-19 2026-08-07              30        1.583333            0.2479

## 5. The pre-registered primary cell

`chg`, `lead_w = 0`, `horizon 4 weeks`, `threshold 0` (always-on, which is what
the question asks for), third deferred SR3, **follow** — pay when the data has
run hot. Written down before the grid ran and reported first whatever the grid
found.

Two books are priced from it. The **discrete** book takes a non-overlapping
four-week position each time the rule fires. The **always-on weekly** book is
the literal reading of "receive when we expect dovish, pay when we expect
hawkish": a position, re-decided every week, that pays a cost only when it
actually turns over — a side change, a contract change, or opening from flat.

In [7]:
PRIM = []
for name, res in RES.items():
    p = res["primary"]
    d, w = p["discrete_score"], p["weekly_score"]
    PRIM.append({
        "sample": name, "cell": p["config"].label(),
        "trades": d["trades"], "avg_bp": d["avg_bp"], "total_bp": d["total_bp"],
        "hit": d["hit"], "SR_ann": d["sharpe_ann"], "t": d["t_stat"],
        "sign_flip_p": (p.get("sign_flip") or {}).get("p"),
        "episodes": int(p["episodes"]["episode"].nunique()) if "episodes" in p else None,
        "wk_weeks": w["weeks"], "wk_total_bp": w["total_bp"],
        "wk_gross_bp": w["gross_total_bp"], "wk_cost_bp": w["turnover_bp"],
        "wk_SR_ann": w["sharpe_ann"], "wk_maxDD_bp": w["max_dd_bp"],
    })
PRIM = pd.DataFrame(PRIM)
print(PRIM.to_string(index=False))

sample                                   cell  trades    avg_bp    total_bp      hit    SR_ann         t  sign_flip_p  episodes  wk_weeks  wk_total_bp  wk_gross_bp  wk_cost_bp  wk_SR_ann  wk_maxDD_bp
   SR3  chg/L0/h4/thr0/out3/follow [lags 0,4]     108  1.412037  152.500000 0.509259  0.181434  0.522948     0.605870        51       432    97.250000   147.000000       49.75   0.116904  -258.500000
 OIS21 chg/L0/h4/thr0/ois2y/follow [lags 0,4]     280 -1.468553 -411.194972 0.460714 -0.227116 -1.054035     0.295885       109      1120   -47.774336    40.475664       88.25  -0.027051  -381.403233
   JPM  chg/L0/h4/thr0/out3/follow [lags 0,4]      30  1.583333   47.500000 0.533333  0.247945  0.376656     0.710914        13       120     0.750000    16.000000       15.25   0.003494  -153.000000


In [8]:
fig = make_subplots(rows=1, cols=len(RES), shared_yaxes=False,
                    subplot_titles=[f"{k}: always-on weekly book" for k in RES])
for i, (name, res) in enumerate(RES.items(), start=1):
    b = res["primary"]["weekly_book"]
    if b is None or b.empty:
        continue
    x = pd.to_datetime(b["exit_date"])
    fig.add_trace(go.Scatter(x=x, y=np.cumsum(b["pnl_bp_gross"].to_numpy(float)),
                             name="gross", line=dict(color=GREY, width=1.4),
                             showlegend=(i == 1)), row=1, col=i)
    fig.add_trace(go.Scatter(x=x, y=np.cumsum(b["pnl_bp"].to_numpy(float)),
                             name="net of turnover", line=dict(color=BLUE, width=2),
                             showlegend=(i == 1)), row=1, col=i)
    fig.add_hline(y=0, line=dict(color=GRID_C, width=1), row=1, col=i)
style(fig, 420, "Cumulative bp -- pay hot data, receive soft data, re-decided weekly")
fig.show()

## 6. The grid, and the null that prices the search

`reading x lead x threshold x horizon x structure`, every cell scored at BOTH
directions inside `best_of_both`, so the trial count is twice the cell count.
The search is paid for by rotating the weekly signal against the calendar and
re-scoring the **entire** grid every time — the observation is a searched
maximum, so every surrogate is scored on the same searched maximum.

The rotation set is finite and is enumerated in full; `min_offset` exceeds
twice the widest alignment searched, so no surrogate can reproduce the true
one.

In [9]:
NULLS = []
for name, res in RES.items():
    n = res["null_summary"]
    NULLS.append({"sample": name, "cells": res["n_cells"], "trials": res["n_trials"],
                  "scored": res["n_scored"],
                  "best_cell": (f"{res['best_cell']['construction']}/"
                                f"L{res['best_cell']['lead_k']}/"
                                f"thr{res['best_cell']['threshold']}/"
                                f"h{res['best_cell']['horizon_w']}/"
                                f"{res['best_cell']['structure']}/"
                                f"{res['best_cell']['sign']}"),
                  "observed": res["best_sharpe"], "null_median": n["median"],
                  "null_q95": n["q95"], "percentile_of_own_null": n["observed_percentile"],
                  "p_rotation": res["p_rotation"], "rotations": n["draws"],
                  "exhaustive": n["exhaustive"], "min_offset": n["min_offset"],
                  "p_floor": n["p_floor"],
                  "DSR": (res.get("deflation") or {}).get("dsr"),
                  "SR0": (res.get("deflation") or {}).get("sr0"),
                  "n_eff_trials": (res.get("deflation") or {}).get("n_eff_evt_mc"),
                  "RW_rejected": (res.get("family") or {}).get("n_rejected"),
                  "RW_tested": (res.get("family") or {}).get("n_tested"),
                  "RW_rank1_p": (res.get("family") or {}).get("rank1_adjusted_p")})
NULLS = pd.DataFrame(NULLS)
print(NULLS.to_string(index=False))
print("\nThe wiring identity: with the FULL family, the Romano-Wolf rank-1 adjusted")
print("p equals the rotation p exactly, because the stepdown's first suffix maximum")
print("IS the grid maximum. Any difference means the two tests are looking at")
print("different families.")
for _, r in NULLS.iterrows():
    same = (abs(float(r["RW_rank1_p"]) - float(r["p_rotation"])) < 1e-12
            if pd.notna(r["RW_rank1_p"]) else None)
    print(f"   {r['sample']:6s} rotation {r['p_rotation']:.4f} vs RW rank-1 "
          f"{r['RW_rank1_p']:.4f}  identical: {same}")

sample  cells  trials  scored                      best_cell  observed  null_median  null_q95  percentile_of_own_null  p_rotation  rotations  exhaustive  min_offset  p_floor      DSR      SR0  n_eff_trials  RW_rejected  RW_tested  RW_rank1_p
   SR3    672    1344     669 chg/L11/thr0.0/h8/pack1/follow  0.108989     0.115490  0.149985                0.335211    0.665730        355        True          39 0.002809 0.008047 0.195687    375.781347            0        669    0.665730
 OIS21     96     192      96 chg/L11/thr0.0/h8/ois2y/follow  0.056144     0.061031  0.083563                0.314176    0.686124       1044        True          39 0.000957 0.270720 0.072620    118.036322            0         96    0.686124
   JPM    288     576     240  chg/L2/thr0.0/h1/ois2y/follow  0.203827     0.206895  0.283145                0.441860    0.568182         43        True          39 0.022727 0.149021 0.294191    147.026561            0        240    0.568182

The wiring identity: with the F

In [10]:
fig = make_subplots(rows=1, cols=len(RES),
                    subplot_titles=[f"{k}: searched max vs its own null" for k in RES])
for i, (name, res) in enumerate(RES.items(), start=1):
    st = np.asarray(res["rotation_null"].get("max_abs_sharpe", []), float)
    if st.size == 0:
        continue
    fig.add_trace(go.Histogram(x=st, nbinsx=30, marker_color=GREY, opacity=0.85,
                               name="rotations", showlegend=(i == 1)), row=1, col=i)
    fig.add_vline(x=float(res["best_sharpe"]), line=dict(color=RED, width=2.5),
                  row=1, col=i)
    fig.add_vline(x=float(res["null_summary"]["median"]),
                  line=dict(color=AMBER, width=1.5, dash="dot"), row=1, col=i)
style(fig, 400, "Red = the grid's best cell. Amber = the null's median.")
fig.show()

The red line sits **left of** the amber one in all three samples. The best cell
a several-hundred-cell search could find is worse than a typical misaligned copy
of the same signal.

## 7. What the league says about the axes

Read the MEDIAN, not the best: a best-of is the maximum of however many cells
that level happens to contain, so ranking axes by their best rewards the axis
with the most cells.

In [11]:
for name, res in RES.items():
    print("=" * 76)
    print(f"{name}   ({res['n_cells']} cells, {res['n_scored']} scored)")
    print("=" * 76)
    lg = res["league"]
    for axis in ("construction", "lead_k", "horizon_w", "structure", "threshold"):
        if lg[axis].nunique() < 2:
            continue
        print(f"\n  by {axis}:")
        print("   " + R.league_summary(lg, axis).to_string().replace("\n", "\n   "))
    print(f"\n  top 5 cells:")
    cols = ["construction", "lead_k", "threshold", "horizon_w", "structure",
            "lag_near_w", "lag_far_w", "sign", "trades", "avg_bp", "sharpe",
            "t_stat", "episodes", "top_episode_share"]
    print("   " + lg.sort_values("sharpe", ascending=False).head(5)[cols]
          .to_string(index=False).replace("\n", "\n   "))

SR3   (672 cells, 669 scored)

  by construction:
                 cells  median_sharpe_wk  best_sharpe_wk  median_avg_bp  follow_share
   construction                                                                      
   chg             333          0.025195        0.108989       1.023148      0.495495
   level           336          0.016824        0.095618       0.614330      0.485119

  by lead_k:
           cells  median_sharpe_wk  best_sharpe_wk  median_avg_bp  follow_share
   lead_k                                                                      
   2         165          0.028652        0.105108       1.107895      0.484848
   0         168          0.027284        0.103686       0.997340      0.482143
   5         168          0.020673        0.099867       0.767411      0.464286
   11        168          0.011160        0.108989       0.436455      0.529762

  by horizon_w:
              cells  median_sharpe_wk  best_sharpe_wk  median_avg_bp  follow_share
   horizon_w

## 8. Sensitivities: the vintage bound, and what a one-week delay does

The Citi surprise snapshot is a **single vintage**. It carries only today's read
of every daily CESI value, Citi revises and periodically rebases, and there is
no publication axis on the surprise side at all — `G1` in the lead study gates
speeches, not data. This is the largest residual look-ahead in the stack and it
cannot be removed here, only bounded: re-run the pre-registered cell with the
composite delayed an extra one and two weeks. A revision published within a week
or two of the observation cannot help a book that is not allowed to see it.

In [12]:
for name, res in RES.items():
    vs = res.get("vintage_sensitivity")
    if vs is None:
        continue
    print(f"{name}: the pre-registered cell with the composite delayed d weeks")
    print("   " + vs.to_string(index=False).replace("\n", "\n   "))
    d0 = vs.loc[vs["extra_delay_w"] == 0].iloc[0]
    rest = vs.loc[vs["extra_delay_w"] > 0]
    print(f"   -> delaying the composite does not DESTROY the result, it MOVES it: "
          f"avg_bp goes {float(d0['avg_bp']):+.3f} -> "
          f"{', '.join(f'{float(x):+.3f}' for x in rest['avg_bp'])}")
    print()
print("A result that flips sign or doubles when the input is delayed by one week")
print("is not a result that a revision could have created -- it is a result that")
print("was never there. That is the honest reading of these three tables.")

SR3: the pre-registered cell with the composite delayed d weeks
    extra_delay_w  trades   avg_bp  gross_avg_bp  total_bp      hit   sharpe  sharpe_ann   t_stat
                0     108 1.412037      1.912037     152.5 0.509259 0.050321    0.181434 0.522948
                1     108 0.560185      1.060185      60.5 0.500000 0.019931    0.071862 0.207129
                2     108 0.495370      0.995370      53.5 0.481481 0.017623    0.063542 0.183148
   -> delaying the composite does not DESTROY the result, it MOVES it: avg_bp goes +1.412 -> +0.560, +0.495

OIS21: the pre-registered cell with the composite delayed d weeks
    extra_delay_w  trades    avg_bp  gross_avg_bp    total_bp      hit    sharpe  sharpe_ann    t_stat
                0     280 -1.468553     -0.968553 -411.194972 0.460714 -0.062991   -0.227116 -1.054035
                1     280  0.708701      1.208701  198.436389 0.492857  0.030413    0.109656  0.508907
                2     280  0.279847      0.779847   78.35725

## 9. Verdict

In [13]:
print("=" * 78)
alive = []
for name, res in RES.items():
    p = res["p_rotation"]
    below = res["best_sharpe"] < res["null_summary"]["median"]
    rw = (res.get("family") or {}).get("n_rejected", 0)
    dsr = (res.get("deflation") or {}).get("dsr", np.nan)
    verdict = "DEAD" if (p > 0.05 or below) else "alive?"
    if verdict != "DEAD":
        alive.append(name)
    print(f"{name:6s}  best {res['best_sharpe']:.4f} vs null median "
          f"{res['null_summary']['median']:.4f}  "
          f"({'BELOW' if below else 'above'})  p_rot {p:.4f}  DSR {dsr:.4f}  "
          f"RW rejects {rw}  -> {verdict}")
print("=" * 78)
print(f"{len(alive)} of {len(RES)} samples alive.")
print()
print("The claim under test was that the economic-surprise composite tells you")
print("which way Fedspeak is about to turn, and that the front end has not")
print("priced it. The first half of that is real and was measured by two prior")
print("studies. The second half is what a trade needs, and across 433 weeks of")
print("SR3, 1,122 weeks of 2y OIS and 121 weeks of the point-in-time sentiment")
print("window, nothing here reaches the price.")
print()
print(f"total notebook runtime {time.time() - T0:.1f}s")

SR3     best 0.1090 vs null median 0.1155  (BELOW)  p_rot 0.6657  DSR 0.0080  RW rejects 0  -> DEAD
OIS21   best 0.0561 vs null median 0.0610  (BELOW)  p_rot 0.6861  DSR 0.2707  RW rejects 0  -> DEAD
JPM     best 0.2038 vs null median 0.2069  (BELOW)  p_rot 0.5682  DSR 0.1490  RW rejects 0  -> DEAD
0 of 3 samples alive.

The claim under test was that the economic-surprise composite tells you
which way Fedspeak is about to turn, and that the front end has not
priced it. The first half of that is real and was measured by two prior
studies. The second half is what a trade needs, and across 433 weeks of
SR3, 1,122 weeks of 2y OIS and 121 weeks of the point-in-time sentiment
window, nothing here reaches the price.

total notebook runtime 1.9s


## Findings tie-out

Every figure the findings block quotes, printed in exactly the form the prose
quotes it, unit included. `_audit_fed_expected_sentiment_numbers.py` requires
each to appear verbatim in an executed cell.

In [14]:
_sr3, _ois, _jpm = RES["SR3"], RES["OIS21"], RES["JPM"]


def _pd(res):      # primary discrete
    return res["primary"]["discrete_score"]


def _pw(res):      # primary weekly
    return res["primary"]["weekly_score"]


def _bc(res):
    b = res["best_cell"]
    return (f"{b['construction']}/L{b['lead_k']}/thr{b['threshold']}/"
            f"h{b['horizon_w']}/{b['structure']}/{b['sign']}")


TIEOUT = {
    "1 SR3 weeks": f"{len(_sr3['support'])}",
    "1 SR3 best cell": _bc(_sr3),
    "1 SR3 best sharpe": f"{_sr3['best_sharpe']:.4f}",
    "1 SR3 null median": f"{_sr3['null_summary']['median']:.4f}",
    "1 SR3 null q95": f"{_sr3['null_summary']['q95']:.4f}",
    "1 SR3 p_rotation": f"{_sr3['p_rotation']:.4f}",
    "1 SR3 percentile of own null": f"{100 * _sr3['null_summary']['observed_percentile']:.1f}%",
    "1 SR3 rotations": f"{_sr3['null_summary']['draws']}",
    "1 SR3 p_floor": f"{_sr3['null_summary']['p_floor']:.4f}",
    "1 SR3 cells": f"{_sr3['n_cells']}",
    "1 SR3 trials": f"{_sr3['n_trials']}",
    "1 SR3 DSR": f"{_sr3['deflation']['dsr']:.4f}",
    "1 SR3 SR0": f"{_sr3['deflation']['sr0']:.4f}",
    "1 SR3 RW rejected": f"{_sr3['family']['n_rejected']}",
    "1 SR3 RW tested": f"{_sr3['family']['n_tested']}",

    "2 OIS weeks": f"{len(_ois['support'])}",
    "2 OIS primary trades": f"{_pd(_ois)['trades']}",
    "2 OIS primary avg_bp": f"{_pd(_ois)['avg_bp']:+.4f}bp",
    "2 OIS primary total_bp": f"{_pd(_ois)['total_bp']:+.2f}bp",
    "2 OIS primary t": f"{_pd(_ois)['t_stat']:.4f}",
    "2 OIS weekly total_bp": f"{_pw(_ois)['total_bp']:+.2f}bp",
    "2 OIS best sharpe": f"{_ois['best_sharpe']:.4f}",
    "2 OIS null median": f"{_ois['null_summary']['median']:.4f}",
    "2 OIS p_rotation": f"{_ois['p_rotation']:.4f}",
    "2 OIS rotations": f"{_ois['null_summary']['draws']}",
    "2 OIS DSR": f"{_ois['deflation']['dsr']:.4f}",

    "3 JPM weeks": f"{len(_jpm['support'])}",
    "3 JPM primary trades": f"{_pd(_jpm)['trades']}",
    "3 JPM primary avg_bp": f"{_pd(_jpm)['avg_bp']:+.4f}bp",
    "3 JPM weekly total_bp": f"{_pw(_jpm)['total_bp']:+.2f}bp",
    "3 JPM best sharpe": f"{_jpm['best_sharpe']:.4f}",
    "3 JPM null median": f"{_jpm['null_summary']['median']:.4f}",
    "3 JPM p_rotation": f"{_jpm['p_rotation']:.4f}",
    "3 JPM rotations": f"{_jpm['null_summary']['draws']}",
    "3 JPM DSR": f"{_jpm['deflation']['dsr']:.4f}",

    "4 roll weeks": f"{ROLL['roll_weeks']}",
    "4 roll total weeks": f"{ROLL['weeks']}",
    "4 naive roll mean": f"{ROLL['naive_roll_mean_bp']:+.2f}bp",
    "4 naive flat mean": f"{ROLL['naive_flat_mean_bp']:+.2f}bp",
    "4 naive roll abs median": f"{ROLL['naive_roll_abs_median_bp']:.1f}bp",
    "4 naive flat abs median": f"{ROLL['naive_flat_abs_median_bp']:.1f}bp",
    "4 true roll mean": f"{ROLL['true_roll_mean_bp']:+.2f}bp",
    "4 fabricated per roll week": f"{ROLL['fabricated_mean_bp']:+.2f}bp",
    "4 fabricated total": f"{ROLL['fabricated_total_bp']:+.1f}bp",
    "4 fabricated on flat weeks": f"{ROLL['fabricated_on_flat_weeks_bp']:.1f}bp",
    "4 worst mark diff anywhere":
        f"{float(pd.to_numeric(GATES['worst_mark_diff'], errors='coerce').max()):.1f}",

    "7 SR3 primary trades": f"{_pd(_sr3)['trades']}",
    "7 SR3 primary avg_bp": f"{_pd(_sr3)['avg_bp']:+.4f}bp",
    "7 SR3 primary total_bp": f"{_pd(_sr3)['total_bp']:+.2f}bp",
    "7 SR3 primary t": f"{_pd(_sr3)['t_stat']:.4f}",
    "7 SR3 primary sign_flip p": f"{_sr3['primary']['sign_flip']['p']:.4f}",
    "7 SR3 primary hit": f"{100 * _pd(_sr3)['hit']:.1f}%",
    "7 SR3 weekly weeks": f"{_pw(_sr3)['weeks']}",
    "7 SR3 weekly total_bp": f"{_pw(_sr3)['total_bp']:+.2f}bp",
    "7 SR3 weekly gross_bp": f"{_pw(_sr3)['gross_total_bp']:+.2f}bp",
    "7 SR3 weekly turnover_bp": f"{_pw(_sr3)['turnover_bp']:.2f}bp",
    "7 SR3 weekly SR_ann": f"{_pw(_sr3)['sharpe_ann']:.4f}",
    "7 SR3 weekly maxDD": f"{_pw(_sr3)['max_dd_bp']:.1f}bp",
    "7 SR3 primary episodes":
        f"{int(_sr3['primary']['episodes']['episode'].nunique())}",

    "6 OIS delay-1 total": f"{float(_ois['vintage_sensitivity'].set_index('extra_delay_w').loc[1, 'total_bp']):+.2f}bp",
    "6 SR3 delay-1 total": f"{float(_sr3['vintage_sensitivity'].set_index('extra_delay_w').loc[1, 'total_bp']):+.2f}bp",
    "6 JPM delay-0 total": f"{float(_jpm['vintage_sensitivity'].set_index('extra_delay_w').loc[0, 'total_bp']):+.2f}bp",
    "6 JPM delay-1 total": f"{float(_jpm['vintage_sensitivity'].set_index('extra_delay_w').loc[1, 'total_bp']):+.2f}bp",
    "6 OIS delay-1 swing": f"{abs(float(_ois['vintage_sensitivity'].set_index('extra_delay_w').loc[1, 'total_bp']) - float(_ois['vintage_sensitivity'].set_index('extra_delay_w').loc[0, 'total_bp'])):.0f}bp",

    "8 SR3 follow share": f"{100 * float((_sr3['league']['sign'] == 'follow').mean()):.1f}%",
    "8 SR3 follow share in the top 50":
        f"{100 * float((_sr3['league'].nlargest(50, 'sharpe')['sign'] == 'follow').mean()):.1f}%",
}
for k, v in TIEOUT.items():
    print(f"  {k:36s} {v}")

  1 SR3 weeks                          433
  1 SR3 best cell                      chg/L11/thr0.0/h8/pack1/follow
  1 SR3 best sharpe                    0.1090
  1 SR3 null median                    0.1155
  1 SR3 null q95                       0.1500
  1 SR3 p_rotation                     0.6657
  1 SR3 percentile of own null         33.5%
  1 SR3 rotations                      355
  1 SR3 p_floor                        0.0028
  1 SR3 cells                          672
  1 SR3 trials                         1344
  1 SR3 DSR                            0.0080
  1 SR3 SR0                            0.1957
  1 SR3 RW rejected                    0
  1 SR3 RW tested                      669
  2 OIS weeks                          1122
  2 OIS primary trades                 280
  2 OIS primary avg_bp                 -1.4686bp
  2 OIS primary total_bp               -411.19bp
  2 OIS primary t                      -1.0540
  2 OIS weekly total_bp                -47.77bp
  2 OIS best sharpe       